# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbottabad123/flyrank-ml-track/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create Week-5 dataset
X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    weights=[0.60, 0.40],
    class_sep=1.2,
    random_state=42
)

X = pd.DataFrame(
    X_raw,
    columns=[f"feature_{i}" for i in range(1, 9)]
)

y = pd.Series(y_raw, name="target")

print("Dataset created successfully.")
print("Rows:", len(X))
print("Features:", X.shape[1])

Dataset created successfully.
Rows: 1000
Features: 8


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# Create grouped validation split

groups = np.arange(len(X)) // 10

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))
print(
    "Groups overlap:",
    bool(
        set(groups[train_idx]).intersection(
            set(groups[test_idx])
        )
    )
)

Training rows: 800
Testing rows: 200
Groups overlap: False


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# Train the validated model

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

print("Model trained successfully.")

Model trained successfully.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# Generate predictions and probabilities

predictions = model.predict(
    X.iloc[test_idx]
)

probabilities = model.predict_proba(
    X.iloc[test_idx]
)[:, 1]

queue = X.iloc[test_idx].copy()

queue["actual"] = y.iloc[test_idx].values
queue["predicted"] = predictions
queue["score"] = probabilities

print("Predictions generated:", len(queue))

Predictions generated: 200


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# Assign reason codes based on model score

def assign_reason(score):

    if score >= 0.80:
        return "HIGH_PRIORITY"

    elif score >= 0.60:
        return "REVIEW_OPPORTUNITY"

    elif score >= 0.40:
        return "UNCERTAIN_REVIEW"

    else:
        return "LOW_PRIORITY"


queue["reason_code"] = queue["score"].apply(
    assign_reason
)

print("Reason codes assigned.")

display(
    queue[
        ["score", "reason_code"]
    ].head(10).round(3)
)

Reason codes assigned.


,score,reason_code
0,0.132,LOW_PRIORITY
1,0.069,LOW_PRIORITY
2,0.712,REVIEW_OPPORTUNITY
3,0.615,REVIEW_OPPORTUNITY
4,0.974,HIGH_PRIORITY
5,0.836,HIGH_PRIORITY
6,0.902,HIGH_PRIORITY
7,0.399,LOW_PRIORITY
8,0.021,LOW_PRIORITY
9,0.564,UNCERTAIN_REVIEW


# Week-8 Showcase — 5-Minute Demo Outline

## 1. Question — ~45 seconds

Can a machine-learning ranking model help FlyRank content teams prioritize existing pages for refresh review?

The practical problem is that a large content backlog can contain many pages showing different levels of observed decline. The goal is to help a reviewer decide which pages deserve attention first.

## 2. Method — ~1 minute

I used an anonymized FlyRank content-refresh dataset containing 30,000 rows.

The target was `is_declining_label`.

I compared four approaches:

- Rule-based baseline
- Decision tree
- Logistic regression
- Random forest

The evaluation used a client-holdout validation design.

The main ranking metric was **Precision@50**, because the practical use case is prioritizing a small number of pages at the top of the review queue.

## 3. One Chart — ~1 minute

**Chart: Precision@50 Model Comparison**

The main comparison is:

- Baseline Rules: 0.240
- Decision Tree: 0.540
- Logistic Regression: 0.400
- Random Forest: 0.740

The random forest was selected because it produced the strongest Precision@50 result.

## 4. One Honest Result — ~1 minute

The random forest achieved **0.740 Precision@50**, compared with **0.240 for the baseline rules**.

Average precision was **0.618** for the random forest compared with **0.468** for the baseline.

This is an observed validation result. It does not prove that using the model will increase rankings, clicks, traffic, or conversions.

## 5. One Recommendation — ~1 minute

Use the model as a **review-prioritization tool**.

A content team can start with the highest-ranked pages, inspect the supporting signals and reason codes, and then make the final editorial decision manually.

The model should not automatically publish, rewrite, delete, or otherwise change content.

### Final Takeaway

The main value of the project is reducing the size of the content backlog that a reviewer needs to investigate first.

**Large backlog → ranked queue → human review → editorial decision**

# Shareable Cuts

## Short Social Post

I built a machine-learning workflow for FlyRank's content-refresh prioritization problem.

Using an anonymized dataset of 30,000 content records, I compared a rule-based baseline with decision tree, logistic regression, and random forest models using a client-holdout validation design.

The selected random forest achieved **0.740 Precision@50** compared with **0.240 for the baseline**, showing a stronger ranking signal for identifying pages associated with observed decline.

The workflow is designed as decision support rather than a claim about Google's ranking algorithm or a guarantee of improved search performance.

---

## Employer-Facing Summary

I built a machine-learning content-refresh prioritization workflow using an anonymized FlyRank dataset of 30,000 content records, comparing multiple models against a rule-based baseline with client-holdout validation. The selected random forest achieved **0.740 Precision@50** compared with **0.240 for the baseline**, providing a stronger signal for prioritizing pages associated with observed decline. I turned the model output into a reviewer-facing ranked queue with reason codes while keeping final content decisions with human reviewers.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.